# 📚 Media Stack API – Search Guide: Renewable Energy in EU Region

This guide shows how to use the [Media Stack API](https://mediastack.com/) to search for news articles that mention **renewable energy** in the **European Union (EU)**.

## 🔑 API Access

To use the Media Stack API, you need an **API key**. You can get one by signing up at [https://mediastack.com/](https://mediastack.com/).

## ✅ Query Parameters

| Parameter    | Description                                   | Example                      |
| ------------ | --------------------------------------------- | ---------------------------- |
| `access_key` | Your API key                                  | `your_api_key_here`          |
| `keywords`   | Search query                                  | `renewable energy`           |
| `languages`  | Limit to EU languages (e.g., en, fr, de, etc) | `en,de,fr,it,es,nl,pl`       |
| `countries`  | Limit to EU countries                         | `fr,de,it,es,nl,pl,se,dk,fi` |
| `sort`       | Sort results by relevance or date             | `published_desc`             |
| `limit`      | Number of results to return                   | `50`                         |

## 📘 Full Request Example

curl -G 'https://api.mediastack.com/v1/news' \
    --data-urlencode 'access_key=your_api_key_here' \
    --data-urlencode 'keywords=renewable energy' \
    --data-urlencode 'languages=en,de,fr,it,es,nl,pl' \
    --data-urlencode 'countries=fr,de,it,es,nl,pl,se,dk,fi' \
    --data-urlencode 'sort=published_desc' \
    --data-urlencode 'limit=50'

## 📚 The allowed categories
- `general` – Uncategorized News
- `business` – Business News
- `entertainment` – Entertainment News
- `health` – Health News
- `science` – Science News
- `sports` – Sports News
- `technology` – Technology News

## 📡 Results

The results are generally good
- Some of the links return articles that do not exist in the websites
- Some of the websites require subscription [Chikago news](https://www.chicagotribune.com)
- Some websites with good data [beblog](https://beblog.net/en/?s=renewable)
- Business websites that show stock markets [American banking](https://www.americanbankingnews.com/)

## 😎 Conclusion(Joseph)

The api returns good results and from its api we can a free trial period and from the different urls we can still build parsers
for very good sources like the ones named above


## 🧪 Testing

For testing create a new .env file in the root directory
create a account in the media stack website and copy the api key
into the .env file

```
MEDIA_STACK_API_KEY=a21711ee42367f6f0215814a320892bb
```


In [3]:
import os
import http.client
import urllib.parse
import json
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

MEDIA_STACK_API_KEY = os.getenv("MEDIA_STACK_API_KEY")

conn = http.client.HTTPConnection('api.mediastack.com')

params = urllib.parse.urlencode({
    'access_key': MEDIA_STACK_API_KEY,
    'categories': '-science,-technology,-business',
    'sort': 'published_asc',
    'keywords': 'renewable energy',
    'limit': 10,
})

conn.request('GET', '/v1/news?{}'.format(params))

res = conn.getresponse()
data = res.read()

# Convert byte response to JSON
json_data = json.loads(data.decode('utf-8'))

# Extract articles and build the source-to-url mapping
source_url_map = {article['source']: article['url'] for article in json_data.get('data', [])}

# Print the result
print(json.dumps(source_url_map, indent=2))

{
  "Chicago Tribune": "https://www.chicagotribune.com/2025/01/31/illinois-clean-energy-leaders-strategize-in-wake-of-trump-orders-make-this-about-consumers/",
  "americanbankingnews": "https://www.americanbankingnews.com/2025/02/01/cormark-has-lowered-expectations-for-innergex-renewable-energy-tseine-stock-price.html",
  "etfdailynews": "https://www.etfdailynews.com/2025/02/01/scotiabank-issues-pessimistic-forecast-for-innergex-renewable-energy-tseine-stock-price/",
  "businesstoday": "https://www.businesstoday.in/markets/stocks/story/budget-2025-26-allocation-to-renewable-energy-jumps-53-heres-what-analysts-say-about-suzlon-waaree-adani-others-463022-2025-02-01?utm_source=rssfeed",
  "lanoticiadigital": "https://lanoticiadigital.com.ar/news-en/shockingly-profitable-discover-the-seven-ev-stocks-investors-cant-ignore/105716/",
  "lulegacy": "https://www.marketbeat.com/instant-alerts/best-renewable-energy-stocks-to-follow-today-january-31st-2025-01-31/",
  "deccanchronicle": "https://ww

### 📰 Using [newspaper](https://pypi.org/project/newspaper/) lib and selenium

In [4]:
import json
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from newspaper import Article

def scrape_with_newspaper_selenium(source_url_map):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("user-agent=Mozilla/5.0")

    results = []

    for source, url in source_url_map.items():
        try:
            driver = webdriver.Chrome(options=options)
            driver.get(url)
            time.sleep(3)

            html = driver.page_source
            driver.quit()

            article = Article(url)
            article.set_html(html)
            article.download_state = 2  # Force state to SUCCESS
            article.parse()

            result = {
                "source": source,
                "title": article.title,
                "url": url,
                "authors": article.authors,
                "published_date": article.publish_date.isoformat() if article.publish_date else None,
                "paragraphs": [p.strip() for p in article.text.split("\n") if p.strip()]
            }

            results.append(result)

        except Exception as e:
            print(f"Error scraping {url} from {source}: {e}")

    print(json.dumps(results, indent=2))

In [5]:
scrape_with_newspaper_selenium(source_url_map)


[
  {
    "source": "Chicago Tribune",
    "title": "Illinois clean energy leaders strategize in wake of Trump orders: \u2018Make this about consumers\u2019",
    "url": "https://www.chicagotribune.com/2025/01/31/illinois-clean-energy-leaders-strategize-in-wake-of-trump-orders-make-this-about-consumers/",
    "authors": [
      "Nara Schoenberg"
    ],
    "published_date": "2025-01-31T00:00:00",
    "paragraphs": [
      "A roundtable hosted by U.S. Rep. Sean Casten on Friday offered a sneak peek at strategies Illinois clean energy supporters are considering as President Donald Trump threatens to reduce funding for wind farms, solar power and electric vehicles.",
      "Casten highlighted the rapid decline of coal in the U.S., the rise of low-cost renewable energy and consumer interest in electric cars, the fastest-growing s ..."
    ]
  },
  {
    "source": "americanbankingnews",
    "title": "- American Market News",
    "url": "https://www.americanbankingnews.com/2025/02/01/cormark

### 📰 Using news beautiful soap and selenium

In [6]:
from bs4 import BeautifulSoup

# Custom headers and cookies
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

COOKIES = {
    'name_of_the_consent_cookie': 'value_indicating_consent',
}

def scrape_articles_with_selenium_bs4(source_url_map):
    results = []

    for source, url in source_url_map.items():
        try:
            options = Options()
            options.add_argument("--headless")
            options.add_argument(f"user-agent={HEADERS['User-Agent']}")

            driver = webdriver.Chrome(options=options)
            driver.get(url)

            # Inject cookies if needed
            for name, value in COOKIES.items():
                driver.add_cookie({'name': name, 'value': value, 'domain': '.' + url.split('/')[2]})

            driver.get(url)  # Reload page with cookies
            time.sleep(3)

            html = driver.page_source
            driver.quit()

            soup = BeautifulSoup(html, "html.parser")

            # Extract title
            title_tag = soup.find("h1") or soup.title
            title = title_tag.get_text(strip=True) if title_tag else "N/A"

            # Extract paragraphs
            paragraphs = [p.get_text(strip=True) for p in soup.find_all("p") if p.get_text(strip=True)]

            # Extract author
            author_tag = soup.find("meta", {"name": "author"})
            author = author_tag["content"] if author_tag else "Unknown"

            # Extract published date
            published_date = None
            date_meta = soup.find("meta", {"property": "article:published_time"}) or \
                        soup.find("meta", {"name": "date"}) or \
                        soup.find("meta", {"itemprop": "datePublished"}) or \
                        soup.find("time")

            if date_meta:
                if date_meta.has_attr("content"):
                    published_date = date_meta["content"]
                elif date_meta.has_attr("datetime"):
                    published_date = date_meta["datetime"]
                else:
                    published_date = date_meta.get_text(strip=True)

            result = {
                "source": source,
                "title": title,
                "url": url,
                "author": author,
                "published_date": published_date,
                "paragraphs": paragraphs
            }

            results.append(result)

        except Exception as e:
            print(f"Error scraping {url} ({source}): {e}")

    print(json.dumps(results, indent=2))


In [7]:
scrape_articles_with_selenium_bs4(source_url_map)


Error scraping https://lanoticiadigital.com.ar/news-en/shockingly-profitable-discover-the-seven-ev-stocks-investors-cant-ignore/105716/ (lanoticiadigital): Message: invalid cookie domain: Cookie 'domain' mismatch
  (Session info: chrome=136.0.7103.114)
Stacktrace:
0   chromedriver                        0x000000010b0c8898 chromedriver + 5986456
1   chromedriver                        0x000000010b0bf9ca chromedriver + 5949898
2   chromedriver                        0x000000010ab78600 chromedriver + 415232
3   chromedriver                        0x000000010ac29633 chromedriver + 1140275
4   chromedriver                        0x000000010abf0312 chromedriver + 906002
5   chromedriver                        0x000000010ac17566 chromedriver + 1066342
6   chromedriver                        0x000000010abf00e3 chromedriver + 905443
7   chromedriver                        0x000000010abbc61d chromedriver + 693789
8   chromedriver                        0x000000010abbd281 chromedriver + 696961
9 

### 📡 Using requests and trafitula

In [8]:
import json
import requests
import trafilatura

def scrape_multiple_with_trafilatura(source_url_map):
    results = []

    for source, url in source_url_map.items():
        try:
            # Fetch the raw HTML from the URL
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            html = response.text
        except Exception as e:
            print(f"Failed to fetch the page from {url} ({source}): {e}")
            continue

        # Extract structured content with metadata
        downloaded = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False,
            output_format='json'
        )

        if downloaded is None:
            print(f"Failed to extract content from {url} ({source})")
            continue

        data = json.loads(downloaded)

        # Extract clean paragraphs
        paragraphs = data.get('text', '').split('\n') if 'text' in data else []

        result = {
            "source": source,
            "title": data.get("title", "Unknown"),
            "url": url,
            "authors": data.get("author", []),
            "published_date": data.get("date", "Unknown"),
            "paragraphs": [p.strip() for p in paragraphs if p.strip()]
        }

        results.append(result)

    print(json.dumps(results, indent=2, ensure_ascii=False))

In [9]:
scrape_multiple_with_trafilatura(source_url_map)

Failed to fetch the page from https://www.americanbankingnews.com/2025/02/01/cormark-has-lowered-expectations-for-innergex-renewable-energy-tseine-stock-price.html (americanbankingnews): 404 Client Error: Not Found for url: https://www.americanbankingnews.com/2025/02/01/cormark-has-lowered-expectations-for-innergex-renewable-energy-tseine-stock-price.html
Failed to fetch the page from https://www.etfdailynews.com/2025/02/01/scotiabank-issues-pessimistic-forecast-for-innergex-renewable-energy-tseine-stock-price/ (etfdailynews): 404 Client Error: Not Found for url: https://www.etfdailynews.com/2025/02/01/scotiabank-issues-pessimistic-forecast-for-innergex-renewable-energy-tseine-stock-price/
[
  {
    "source": "Chicago Tribune",
    "title": "Unknown",
    "url": "https://www.chicagotribune.com/2025/01/31/illinois-clean-energy-leaders-strategize-in-wake-of-trump-orders-make-this-about-consumers/",
    "authors": [],
    "published_date": "Unknown",
    "paragraphs": [
      "A roundtabl